In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

text = """
The cat sat on the mat.
The cat ate the fish.
The fish sat near the cat.
The cat looked at the fish.
The fish ran away.
The cat sat quietly on the mat.
""" * 100

print(text[:300])
print("Characters:", len(text))


The cat sat on the mat.
The cat ate the fish.
The fish sat near the cat.
The cat looked at the fish.
The fish ran away.
The cat sat quietly on the mat.

The cat sat on the mat.
The cat ate the fish.
The fish sat near the cat.
The cat looked at the fish.
The fish ran away.
The cat sat quietly on the
Characters: 15300


In [3]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(text):
    return [stoi[ch] for ch in text]

def decode(ids):
    return ''.join(itos[i] for i in ids)

data = torch.tensor(
    encode(text),
    dtype=torch.long
)

print("Vocabulary:", chars)
print("Vocab size:", vocab_size)

Vocabulary: ['\n', ' ', '.', 'T', 'a', 'c', 'd', 'e', 'f', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'q', 'r', 's', 't', 'u', 'w', 'y']
Vocab size: 23


In [4]:
split = int(0.9 * len(data))

train_data = data[:split]
val_data = data[split:]

print("Train:", len(train_data))
print("Validation:", len(val_data))

Train: 13770
Validation: 1530


In [5]:
batch_size = 32
block_size = 32

def get_batch(data):

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x, y

In [6]:
x, y = get_batch(train_data)

print("X:", x.shape)
print("Y:", y.shape)

print("Input:")
print(decode(x[0].tolist()))

print("\nTarget:")
print(decode(y[0].tolist()))

X: torch.Size([32, 32])
Y: torch.Size([32, 32])
Input:
way.
The cat sat quietly on the 

Target:
ay.
The cat sat quietly on the m


In [7]:
class TransformerBlock(nn.Module):

    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            d_model,
            num_heads,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x, mask):

        attn_output, _ = self.attention(
            x, x, x,
            attn_mask=mask
        )

        x = self.norm1(x + attn_output)

        x = self.norm2(
            x + self.ffn(x)
        )

        return x

In [8]:
class GPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        block_size,
        d_model=64,
        num_heads=4,
        num_layers=2,
        d_ff=256
    ):
        super().__init__()

        self.block_size = block_size

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.position_embedding = nn.Embedding(
            block_size,
            d_model
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model,
                num_heads,
                d_ff
            )
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)

        self.lm_head = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, idx, targets=None):

        B, T = idx.shape

        positions = torch.arange(
            T,
            device=idx.device
        )

        x = (
            self.token_embedding(idx)
            + self.position_embedding(positions)
        )

        mask = torch.triu(
            torch.ones(
                T, T,
                device=idx.device
            ),
            diagonal=1
        ).bool()

        for block in self.blocks:
            x = block(x, mask)

        x = self.norm(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits_flat = logits.view(
                B * T,
                C
            )

            targets_flat = targets.view(
                B * T
            )

            loss = F.cross_entropy(
                logits_flat,
                targets_flat
            )

        return logits, loss

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = GPT(
    vocab_size=vocab_size,
    block_size=block_size
).to(device)

print("Device:", device)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

Device: cpu
Parameters: 105111


In [10]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

In [11]:
max_steps = 1000

for step in range(max_steps):

    x, y = get_batch(train_data)

    x = x.to(device)
    y = y.to(device)

    logits, loss = model(x, y)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if step % 100 == 0:
        print(
            f"Step {step}: Loss {loss.item():.4f}"
        )

Step 0: Loss 3.3715
Step 100: Loss 1.2970
Step 200: Loss 0.7565
Step 300: Loss 0.3697
Step 400: Loss 0.2114
Step 500: Loss 0.1630
Step 600: Loss 0.1146
Step 700: Loss 0.1100
Step 800: Loss 0.0888
Step 900: Loss 0.0878


In [12]:
@torch.no_grad()
def estimate_loss(data):

    losses = []

    for _ in range(20):

        x, y = get_batch(data)

        x = x.to(device)
        y = y.to(device)

        _, loss = model(x, y)

        losses.append(loss.item())

    return sum(losses) / len(losses)

In [13]:
print("Train loss:",
      estimate_loss(train_data))

print("Validation loss:",
      estimate_loss(val_data))

Train loss: 0.09125373661518096
Validation loss: 0.09122908413410187


In [14]:
torch.save(
    model.state_dict(),
    "tiny_gpt.pt"
)

print("Model saved!")

Model saved!


In [15]:
@torch.no_grad()
def generate(model, idx, max_new_tokens):

    model.eval()

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -block_size:]

        logits, _ = model(idx_cond)

        logits = logits[:, -1, :]

        probs = F.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    return idx

In [16]:
prompt = "The cat"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long
).to(device)

generated = generate(
    model,
    context,
    max_new_tokens=100
)

print(
    decode(generated[0].tolist())
)

The cat sat quietly on the mat.

The cat sat on the mat.
The cat ate the fish.
The fish rat near the cat.
T


In [17]:
@torch.no_grad()
def generate_greedy(model, idx, max_new_tokens):

    model.eval()

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -block_size:]

        logits, _ = model(idx_cond)

        logits = logits[:, -1, :]

        next_token = torch.argmax(
            logits,
            dim=-1,
            keepdim=True
        )

        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    return idx

In [18]:
generated = generate_greedy(
    model,
    context,
    100
)

print(
    decode(generated[0].tolist())
)

The cat sat on the mat.
The cat ate the fish.
The fish sat near the cat.
The cat looked at the fish.
The fi


In [19]:
@torch.no_grad()
def generate_temperature(
    model,
    idx,
    max_new_tokens,
    temperature=1.0
):

    model.eval()

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -block_size:]

        logits, _ = model(idx_cond)

        logits = logits[:, -1, :]

        logits = logits / temperature

        probs = F.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            1
        )

        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    return idx

In [20]:
for temperature in [0.2, 0.7, 1.2]:

    generated = generate_temperature(
        model,
        context,
        80,
        temperature
    )

    print("\nTemperature:", temperature)
    print(decode(generated[0].tolist()))


Temperature: 0.2
The cat looked at the fish.
The fish ran away.
The cat sat quietly on the mat.

The cat

Temperature: 0.7
The cat looked at the fish.
The fish ran away.
The cat sat quietly on the mat.

The cat

Temperature: 1.2
The cat sat quietly on the mat.

The cat sat on the mat.
The cat ate the fish.
The fish


In [21]:
@torch.no_grad()
def generate_top_k(
    model,
    idx,
    max_new_tokens,
    temperature=1.0,
    top_k=5
):

    model.eval()

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -block_size:]

        logits, _ = model(idx_cond)

        logits = logits[:, -1, :]

        logits = logits / temperature

        values, indices = torch.topk(
            logits,
            top_k
        )

        filtered = torch.full_like(
            logits,
            float("-inf")
        )

        filtered.scatter_(
            1,
            indices,
            values
        )

        probs = F.softmax(
            filtered,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            1
        )

        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    return idx

In [22]:
generated = generate_top_k(
    model,
    context,
    100,
    temperature=0.8,
    top_k=5
)

print(
    decode(generated[0].tolist())
)

The cat sat on the mat.
The cat ate the fish.
The fish sat near the cat.
The cat looked at the fish.
The fi


In [23]:
@torch.no_grad()
def generate_top_p(
    model,
    idx,
    max_new_tokens,
    temperature=1.0,
    top_p=0.9
):

    model.eval()

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -block_size:]

        logits, _ = model(idx_cond)

        logits = logits[:, -1, :]

        logits = logits / temperature

        probs = F.softmax(
            logits,
            dim=-1
        )

        sorted_probs, sorted_indices = torch.sort(
            probs,
            descending=True
        )

        cumulative_probs = torch.cumsum(
            sorted_probs,
            dim=-1
        )

        remove = cumulative_probs > top_p

        remove[:, 1:] = remove[:, :-1].clone()
        remove[:, 0] = False

        sorted_probs[remove] = 0

        sorted_probs = sorted_probs / sorted_probs.sum(
            dim=-1,
            keepdim=True
        )

        next_sorted = torch.multinomial(
            sorted_probs,
            1
        )

        next_token = sorted_indices.gather(
            -1,
            next_sorted
        )

        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    return idx

In [24]:
generated = generate_top_p(
    model,
    context,
    100,
    temperature=0.8,
    top_p=0.9
)

print(
    decode(generated[0].tolist())
)

The cat looked at the fish.
The fish ran away.
The cat sat quietly on the mat.

The cat sat on the mat.
The
